In [ ]:
import os
import numpy as np
import pandas as pd

def process_file(tree_id, vs, input_dir, output_dir):
    """
    Processes a single file for the given tree_id and voxel size:
      1. Loads the CSV file.
      2. Adds extra columns.
      3. Recalculates CI_lw and CI_leaf using the exact G values.
      4. Computes corrected PAD and LAD columns using the new CI values.
      5. Groups the numeric data by voxel center coordinates (voxel_cx, voxel_cy, voxel_cz) using the mean.
      6. Saves the grouped data to a new CSV file with the name pattern:
         {tree_id}_grouped_voxel_size_{vs:.1f}.csv
    """
    input_file = os.path.join(input_dir, f"{tree_id}_all_details_voxel_size_{vs:.1f}.csv")
    if not os.path.exists(input_file):
        print(f"Warning: File not found - {input_file}")
        return

    # Load the CSV file
    df = pd.read_csv(input_file)
    df['voxel_size'] = vs
    df['tree_id'] = int(tree_id)
    # Count unique voxels (for reference, not used later)
    df['num_voxels'] = len(np.unique(df[['voxel_cx', 'voxel_cy', 'voxel_cz']], axis=0))
    
    # --- Recalculate the canopy openness (CI) values using the exact G values ---
    # (Existing CI values were computed with G = 0.5, so correct them by multiplying by 0.5/G_exact)
    df["CI_lw_corr"] = df["CI_lw"] * (0.5 / df["G_lw_computed"])
    df["CI_leaf_corr"] = df["CI_leaf"] * (0.5 / df["G_leaf_computed"])
    
    # --- Compute corrected PAD columns (using G_lw_computed and the corrected CI_lw_corr) ---
    df["PAD_BL_G05"]    = 0.5 * df["PAD_BL"]     / df["G_lw_computed"]
    df["PAD_BL_CI1"]    = df["PAD_BL"]           / df["CI_lw_corr"]
    ratio_pad_CI_G      = df["G_lw_computed"] * df["CI_lw_corr"]
    df["PAD_BL_G05_CI1"] = 0.5 * df["PAD_BL"]    / ratio_pad_CI_G

    df["PAD_BL_EPL_G05"]    = 0.5 * df["PAD_BL_EPL"]     / df["G_lw_computed"]
    df["PAD_BL_EPL_CI1"]    = df["PAD_BL_EPL"]           / df["CI_lw_corr"]
    df["PAD_BL_EPL_G05_CI1"] = 0.5 * df["PAD_BL_EPL"]    / ratio_pad_CI_G

    df["PAD_BL_UEPL_G05"]    = 0.5 * df["PAD_BL_UEPL"]     / df["G_lw_computed"]
    df["PAD_BL_UEPL_CI1"]    = df["PAD_BL_UEPL"]           / df["CI_lw_corr"]
    df["PAD_BL_UEPL_G05_CI1"] = 0.5 * df["PAD_BL_UEPL"]    / ratio_pad_CI_G

    df["PAD_MCF_G05"]    = 0.5 * df["PAD_MCF"]     / df["G_lw_computed"]
    df["PAD_MCF_CI1"]    = df["PAD_MCF"]           / df["CI_lw_corr"]
    df["PAD_MCF_G05_CI1"] = 0.5 * df["PAD_MCF"]    / ratio_pad_CI_G

    df["PAD_MCF_Corr_G05"]    = 0.5 * df["PAD_MCF_Corr"]     / df["G_lw_computed"]
    df["PAD_MCF_Corr_CI1"]    = df["PAD_MCF_Corr"]           / df["CI_lw_corr"]
    df["PAD_MCF_Corr_G05_CI1"] = 0.5 * df["PAD_MCF_Corr"]    / ratio_pad_CI_G

    df["PAD_MLE"]          = df["PAD_MLE_pimont_2018"] / (df["leaf_fraction"] * df["alpha"])
    df["PAD_MLE_G05"]      = 0.5 * df["PAD_MLE"]       / df["G_lw_computed"]
    df["PAD_MLE_CI1"]      = df["PAD_MLE"]             / df["CI_lw_corr"]
    df["PAD_MLE_G05_CI1"]  = 0.5 * df["PAD_MLE"]       / ratio_pad_CI_G

    # --- Compute corrected LAD columns (using G_leaf_computed and the corrected CI_leaf_corr) ---
    ratio_lad_CI_G = df["G_leaf_computed"] * df["CI_leaf_corr"]

    df["LAD_BL_G05"]       = 0.5 * df["LAD_BL"]       / df["G_leaf_computed"]
    df["LAD_BL_CI1"]       = df["LAD_BL"]             / df["CI_leaf_corr"]
    df["LAD_BL_G05_CI1"]   = 0.5 * df["LAD_BL"]       / ratio_lad_CI_G

    df["LAD_BL_EPL_G05"]   = 0.5 * df["LAD_BL_EPL"]   / df["G_leaf_computed"]
    df["LAD_BL_EPL_CI1"]   = df["LAD_BL_EPL"]         / df["CI_leaf_corr"]
    df["LAD_BL_EPL_G05_CI1"] = 0.5 * df["LAD_BL_EPL"] / ratio_lad_CI_G

    df["LAD_BL_UEPL_G05"]   = 0.5 * df["LAD_BL_UEPL"]   / df["G_leaf_computed"]
    df["LAD_BL_UEPL_CI1"]   = df["LAD_BL_UEPL"]         / df["CI_leaf_corr"]
    df["LAD_BL_UEPL_G05_CI1"] = 0.5 * df["LAD_BL_UEPL"] / ratio_lad_CI_G

    df["LAD_MCF_G05"]      = 0.5 * df["LAD_MCF"]       / df["G_leaf_computed"]
    df["LAD_MCF_CI1"]      = df["LAD_MCF"]             / df["CI_leaf_corr"]
    df["LAD_MCF_G05_CI1"]  = 0.5 * df["LAD_MCF"]       / ratio_lad_CI_G

    df["LAD_MCF_Corr_G05"]      = 0.5 * df["LAD_MCF_Corr"]       / df["G_leaf_computed"]
    df["LAD_MCF_Corr_CI1"]      = df["LAD_MCF_Corr"]             / df["CI_leaf_corr"]
    df["LAD_MCF_Corr_G05_CI1"]  = 0.5 * df["LAD_MCF_Corr"]       / ratio_lad_CI_G

    df["LAD_MLE"]          = df["LAD_MLE_pimont_2019"]
    df["LAD_MLE_G05"]     = 0.5 * df["LAD_MLE"] / df["G_leaf_computed"]
    df["LAD_MLE_CI1"]     = df["LAD_MLE"]       / df["CI_leaf_corr"]
    df["LAD_MLE_G05_CI1"] = 0.5 * df["LAD_MLE"] / ratio_lad_CI_G

    # --- Group the data by voxel center coordinates ---
    group_cols = ['voxel_cx', 'voxel_cy', 'voxel_cz']
    # Select only numeric columns to avoid errors with non-numeric data
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # Ensure the grouping columns are included in the numeric selection
    for col in group_cols:
        if col not in numeric_cols:
            numeric_cols.append(col)
    df_numeric = df[numeric_cols]
    df_grouped = df_numeric.groupby(group_cols, as_index=False).mean()

    # Save the grouped data with the desired filename pattern
    output_file = os.path.join(output_dir, f"{tree_id}_grouped_voxel_size_{vs:.1f}.csv")
    df_grouped.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")

if __name__ == '__main__':
    # Define tree IDs and voxel sizes
    tree_ids = ["003", "004", "006", "007", "008", "009", "010", "011", "012", "013", "014", "015", "016", "017", "018", "023", "024", "025", "026"]
    voxel_sizes = [0.2, 0.5, 1.0, 1.5, 2.0]

    # Directories for input and output files
    input_dir = "/QRISdata/Q5866/uqrarya1/result_analysis/voxel_all_metrics_references"
    output_dir = "grouped_voxel_data"
    os.makedirs(output_dir, exist_ok=True)

    # Process each file and save the grouped output
    for tree_id in tree_ids:
        for vs in voxel_sizes:
            process_file(tree_id, vs, input_dir, output_dir)
